# Competencia — Aprendizaje de Máquina 2026-10
## Parte 1: Clasificación de Textos con Machine Learning Clásico

Este notebook implementa el modelo final de clasificación de textos históricos en español/latín
según su **década de origen** (39 clases, de 1500 a 1880). El enfoque combina lo mejor de
dos líneas de experimentación independientes del equipo:

- **Char TF-IDF (1,4) con 400k features + 41 features lingüísticas:** configuración ganadora
  encontrada tras búsqueda exhaustiva de hiperparámetros en versiones anteriores → **Kaggle: 0.290**
- **Word TF-IDF (1,2) con 100k features:** representación complementaria que captura vocabulario
  y bigramas históricos → aporte que subió el score a **Kaggle: 0.306**

La combinación de ambas representaciones más las features numéricas históricas produce una
matriz final de **500.041 features** clasificada con **LinearSVC**.

## 1. Importaciones

Se importan las librerías necesarias y se definen las constantes globales del experimento.
Los valores `BEST_ANALYZER`, `BEST_NGRAM` y `BEST_MAXF` corresponden a la configuración
ganadora validada en versiones anteriores del notebook y se mantienen fijos en este archivo.

In [2]:
import pandas as pd
import numpy as np
import re, os, math, warnings
from collections import Counter
warnings.filterwarnings('ignore')

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.svm import LinearSVC
from sklearn.feature_selection import f_classif
from sklearn.metrics import f1_score
from scipy.sparse import hstack, csr_matrix
import matplotlib.pyplot as plt

SEED = 42
os.makedirs('./submissions', exist_ok=True)

# Confirmado de v4
BEST_ANALYZER  = 'char'
BEST_NGRAM     = (1, 4)
BEST_MAXF      = 400000
BASELINE_LOCAL  = 0.2737   # mejor de v4
BASELINE_KAGGLE = 0.2900   # mejor Kaggle actual

print('✅ Listo')

✅ Listo


## 2. Carga de Datos

El dataset contiene textos históricos en español y latín distribuidos en dos archivos:
- `train.csv`: 31.403 textos etiquetados con su década de origen
- `eval.csv`: 3.490 textos sin etiqueta sobre los cuales se generan las predicciones finales

La variable objetivo `decade` representa los tres primeros dígitos del año (ej: `157` para
la década de 1570), dando un total de **39 clases** desde 150 hasta 188.

In [3]:
df_train = pd.read_csv('./Data/train.csv')
df_eval  = pd.read_csv('./Data/eval.csv')
print(f'Train: {df_train.shape} | Eval: {df_eval.shape}')

Train: (31403, 2) | Eval: (3490, 2)


## 3. Preprocesamiento — Normalización OCR

Los textos provienen de digitalización de documentos históricos y contienen artefactos
tipográficos y de OCR que deben normalizarse antes de vectorizar. La función `normalize_ocr`
aplica un mapa de sustituciones (`CHAR_MAP`) que corrige:

- **Ligaduras tipográficas:** `ﬁ→fi`, `ﬂ→fl`, `ﬀ→ff`, `ﬃ→ffi`, `ﬄ→ffl`
- **Caracteres latinos:** `æ→ae`, `œ→oe`
- **Guiones de corte de línea:** `-\n→` vacío
- **Comillas tipográficas:** `»`, `«`, `"`, `"`, `'`, `'` → formas estándar
- **Símbolos OCR frecuentes:** `£`, `§`, `¶`, `†`, `‡`, `•`, `—`, `–`

Se eliminan además los 51 textos duplicados detectados en el EDA, quedando **31.352 textos**
para entrenamiento. La ortografía arcaica (`hazer`, `dize`, `vna`) se preserva
intencionalmente — es señal temporal clave para la clasificación.

In [4]:
# ── Normalización OCR (idéntica a v3/v4) ──
CHAR_MAP = [
    ('\ufb01','fi'),('\ufb02','fl'),('\ufb00','ff'),('\ufb03','ffi'),('\ufb04','ffl'),
    ('\xe6','ae'),('\u0153','oe'),
    ('-\n',''),('- \n',''),('\xad',''),
    ('\xbb',' '),('\xab',' '),
    ('\u2018',"'"),("\u2019","'"),("\u201c",'"'),("\u201d",'"'),
    ('\xa3',' '),('\xa7',' '),('\xb6',' '),
    ('\u2020',' '),('\u2021',' '),('\u2022',' '),
    ('\u2014',' '),('\u2013',' '),
]

def normalize_ocr(text):
    text = str(text)
    for src, tgt in CHAR_MAP:
        text = text.replace(src, tgt)
    text = text.replace('\n',' ').replace('\r',' ').replace('\t',' ')
    return re.sub(r'  +', ' ', text).strip()

data = df_train.drop_duplicates(subset='text').reset_index(drop=True).copy()
data['text_clean']    = data['text'].apply(normalize_ocr)
df_eval['text_clean'] = df_eval['text'].apply(normalize_ocr)
print(f'Train sin duplicados: {len(data)}')

Train sin duplicados: 31352


## 4. Extracción de Features Lingüísticas Históricas

Se extraen **41 features numéricas** que capturan características morfológicas, ortográficas
y tipográficas correlacionadas con la época de los textos. Se organizan en cuatro grupos:

**Morfología de palabras:** longitud media, desviación estándar, percentiles 75 y 90,
ratio de palabras largas (>10 chars), cortas (≤2) y medias (3-6).

**Estructura del texto:** entropía de caracteres, ratio consonantes/vocales, ratio
palabras/caracteres, longitud media de oraciones, número de oraciones.

**Puntuación:** tasas de coma, punto, punto y coma, dos puntos, paréntesis,
exclamación, interrogación y puntuación total.

**Patrones históricos (regex):** uso de `v` como vocal (`vna`, `vn`), terminaciones
latinas (`-orum`, `-ibus`, `-atis`), nominativos latinos (`-um`, `-us`, `-ae`),
formas arcaicas con `qu` (`quando`, `quasi`), dígrafos `ss` y `ff`, terminaciones
`-cion`, `-tion`, `-sion`, abreviaturas, números romanos, diacríticos raros,
mayúsculas mixtas, ratio de mayúsculas, dígitos, letras y palabras en mayúscula.

In [5]:
# ── Features históricas (idénticas a v3/v4, ya validadas) ──
RE_LONG_S    = re.compile(r'[bcdfghjklmnpqrstvwxyz]f[aeiouáéíóú]', re.I)
RE_V_AS_U    = re.compile(r'\bvn[aeiouáéíóú]|\bvn\b|\bvm\b', re.I)
RE_ROMAN     = re.compile(
    r'\b(M{1,4}(CM|CD|D?C{0,3})(XC|XL|L?X{0,3})(IX|IV|V?I{0,3})'
    r'|CM|CD|XC|XL|IX|IV|D?C{2,3}|L?X{2,3}|V?I{2,4})\b')
RE_LATIN_END = re.compile(r'\b\w{3,}(orum|ibus|atis|endi|antis|entis)\b', re.I)
RE_LATIN_NOM = re.compile(r'\b\w{3,}(um|us|ae)\b', re.I)
RE_QU_ARC    = re.compile(r'\bqu[aou]\w', re.I)
RE_SS        = re.compile(r'ss', re.I)
RE_FF        = re.compile(r'ff', re.I)
RE_CION      = re.compile(r'\b\w{3,}cion\b', re.I)
RE_TION      = re.compile(r'\b\w{3,}tion\b', re.I)
RE_SION      = re.compile(r'\b\w{3,}sion\b', re.I)
RE_ABBREV    = re.compile(r'\b[A-Za-z]{1,4}\.')
RE_MCASE     = re.compile(r'\b[A-Z][a-z]{1,}[A-Z]\w*\b')
RE_RDIAC     = re.compile(r'[àâãäāăąèêëēĕěîïīĭôõōŏùûüūŭ]', re.I)
RE_SEMI      = re.compile(r';')
RE_COLON     = re.compile(r':')
RE_PAREN     = re.compile(r'[()]')

def extract_features(text):
    words   = text.split()
    n       = max(len(words), 1)
    nc      = max(len(text), 1)
    lengths = [len(w) for w in words]
    counts  = Counter(text)
    total   = len(text) or 1
    entropy = -sum((c/total)*math.log2(c/total) for c in counts.values()) if text else 0
    vowels  = sum(1 for c in text.lower() if c in 'aeiouáéíóúàèìòùäëïöüâêîôû')
    cons    = sum(1 for c in text.lower() if c.isalpha()
                  and c not in 'aeiouáéíóúàèìòùäëïöüâêîôû')
    sents   = [s.strip() for s in re.split(r'[.!?]+', text) if s.strip()]
    ns      = max(len(sents), 1)
    return {
        'wl_mean':    np.mean(lengths) if lengths else 0,
        'wl_std':     np.std(lengths)  if lengths else 0,
        'wl_p75':     np.percentile(lengths, 75) if lengths else 0,
        'wl_p90':     np.percentile(lengths, 90) if lengths else 0,
        'ratio_long': sum(1 for l in lengths if l > 10) / n,
        'ratio_short':sum(1 for l in lengths if l <= 2) / n,
        'ratio_med':  sum(1 for l in lengths if 3 <= l <= 6) / n,
        'char_entropy':   entropy,
        'cv_ratio':       cons / vowels if vowels > 0 else 0,
        'word_char_ratio':n / nc,
        'comma_rate':     text.count(',') / n,
        'period_rate':    text.count('.') / n,
        'semicolon_rate': len(RE_SEMI.findall(text))  / n,
        'colon_rate':     len(RE_COLON.findall(text)) / n,
        'paren_rate':     len(RE_PAREN.findall(text)) / n,
        'excl_rate':      text.count('!') / n,
        'quest_rate':     text.count('?') / n,
        'total_punct':    sum(1 for c in text if c in '.,;:!?()[]{}') / n,
        'long_s_rate':    len(RE_LONG_S.findall(text))    / n,
        'v_as_u_rate':    len(RE_V_AS_U.findall(text))    / n,
        'latin_case':     len(RE_LATIN_END.findall(text)) / n,
        'latin_nom':      len(RE_LATIN_NOM.findall(text)) / n,
        'qu_archaic':     len(RE_QU_ARC.findall(text))    / n,
        'ss_rate':        len(RE_SS.findall(text))         / n,
        'ff_rate':        len(RE_FF.findall(text))         / n,
        'cion_rate':      len(RE_CION.findall(text))       / n,
        'tion_rate':      len(RE_TION.findall(text))       / n,
        'sion_rate':      len(RE_SION.findall(text))       / n,
        'abbrev_rate':    len(RE_ABBREV.findall(text))     / n,
        'roman_rate':     len(RE_ROMAN.findall(text))      / n,
        'rare_diac':      len(RE_RDIAC.findall(text))      / nc,
        'mixed_case':     len(RE_MCASE.findall(text))      / n,
        'ttr':         len(set(w.lower() for w in words)) / n,
        'upper_ratio': sum(1 for c in text if c.isupper()) / nc,
        'digit_ratio': sum(1 for c in text if c.isdigit()) / nc,
        'alpha_ratio': sum(1 for c in text if c.isalpha()) / nc,
        'all_caps':    sum(1 for w in words if w.isupper() and len(w)>1) / n,
        'pure_digit':  sum(1 for w in words if w.isdigit()) / n,
        'n_words':     float(n),
        'avg_sent_len':n / ns,
        'n_sentences': float(ns),
    }

print('Extrayendo features train...')
feats_train = pd.DataFrame(
    data['text_clean'].apply(extract_features).tolist()
).fillna(0)
print('Extrayendo features eval...')
feats_eval = pd.DataFrame(
    df_eval['text_clean'].apply(extract_features).tolist()
).fillna(0)

ALL_FEATS = feats_train.columns.tolist()
print(f'✅ Features: {len(ALL_FEATS)}')

Extrayendo features train...
Extrayendo features eval...
✅ Features: 41


### 4.1 Selección de Features por Relevancia Estadística

Para identificar cuáles de las 41 features tienen mayor poder discriminativo entre décadas,
aplicamos un **test F de ANOVA** (`f_classif`) que mide la varianza entre clases relativa
a la varianza dentro de cada clase. Las features con mayor F-score son las que más varían
sistemáticamente entre décadas.

In [6]:
# ── Top features por F-score ──
f_scores, _ = f_classif(feats_train.values, data['decade'].values)
df_disc = pd.DataFrame({'feature': ALL_FEATS, 'f_score': f_scores})\
            .sort_values('f_score', ascending=False).reset_index(drop=True)
TOP_10 = df_disc.head(10)['feature'].tolist()
print('Top 10 features:', TOP_10)

Top 10 features: ['wl_std', 'wl_mean', 'ratio_long', 'word_char_ratio', 'wl_p90', 'comma_rate', 'char_entropy', 'long_s_rate', 'colon_rate', 'ratio_short']


Las **top 10 features** identificadas son:
`wl_std`, `wl_mean`, `ratio_long`, `word_char_ratio`, `wl_p90`, `comma_rate`,
`char_entropy`, `long_s_rate`, `colon_rate`, `ratio_short`

Destacan las features de longitud de palabra (`wl_mean`, `wl_std`, `wl_p90`, `ratio_long`)
como las más discriminativas — los textos del s.XVI usan palabras significativamente más
largas que los del s.XIX, reflejando la morfología latina del español antiguo. La
`long_s_rate` captura el uso del dígrafo `sf` típico del español premoderno.

Este ranking se usa en el EJE 4 para comparar si el top-10 supera al conjunto completo.

## 5. Partición de Datos

Se realiza una partición estratificada 80/20 para la fase de exploración de hiperparámetros.
El conjunto de evaluación (`X_eval_full`) se construye concatenando el texto limpio con
las features numéricas, manteniendo la misma estructura que el conjunto de entrenamiento.

La partición usa `stratify=y_all` para garantizar que cada fold tenga representación
proporcional de las 39 décadas — crítico con clases balanceadas de ~800 textos cada una.

- **Train:** 25.081 textos (80%)
- **Test:** 6.271 textos (20%)

In [7]:
# ── Partición ──
X_all = pd.concat([
    data['text_clean'].reset_index(drop=True),
    feats_train.reset_index(drop=True)
], axis=1)
y_all = data['decade'].reset_index(drop=True)

X_eval_full = pd.concat([
    df_eval['text_clean'].reset_index(drop=True),
    feats_eval.reset_index(drop=True)
], axis=1)

X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, test_size=0.2, random_state=SEED, stratify=y_all
)
print(f'Train: {X_train.shape} | Test: {X_test.shape}')

Train: (25081, 42) | Test: (6271, 42)


## 6. Funciones Auxiliares del Pipeline

Se definen tres funciones reutilizables que encapsulan el pipeline de vectorización,
construcción de matrices y evaluación:

- **`make_vec(min_df, max_f)`:** crea el vectorizador char TF-IDF con la configuración
  ganadora (`analyzer='char'`, `ngram_range=(1,4)`, `sublinear_tf=True`, `max_df=0.98`,
  `strip_accents=None`). El parámetro `strip_accents=None` preserva tildes y diacríticos
  que son señal temporal en español histórico.

- **`build_matrices(vec, feat_cols)`:** construye las matrices sparse train/test/eval
  combinando TF-IDF con features numéricas escaladas. Usa `fit_transform` solo en train
  y `transform` en test/eval para evitar fuga de información.

- **`fit_eval(name, mat_tr, mat_te)`:** entrena un LinearSVC y reporta F1-macro con
  delta respecto al baseline. Usa `dual=False` que es 3-5x más rápido cuando
  `n_features >> n_samples` — el caso aquí con 400k features y 25k textos.

In [8]:
# ── Función de construcción de matrices ──
# dual=False es 3-5x más rápido cuando n_features >> n_samples
# max_iter=1000 suficiente para TF-IDF sublinear normalizado

def make_vec(min_df=1, max_f=400000):
    return TfidfVectorizer(
        analyzer=BEST_ANALYZER,
        ngram_range=BEST_NGRAM,
        sublinear_tf=True,
        max_features=max_f,
        min_df=min_df,
        max_df=0.98,
        strip_accents=None,
    )

def build_matrices(vec, feat_cols, fit_vec=True,
                   X_tr=None, X_te=None, X_ev=None):
    """Construye matrices train/test/eval combinando TF-IDF + features numéricas."""
    X_tr = X_tr if X_tr is not None else X_train
    X_te = X_te if X_te is not None else X_test
    X_ev = X_ev if X_ev is not None else X_eval_full

    if fit_vec:
        tfidf_tr = vec.fit_transform(X_tr['text_clean'])
    else:
        tfidf_tr = vec.transform(X_tr['text_clean'])
    tfidf_te = vec.transform(X_te['text_clean'])
    tfidf_ev = vec.transform(X_ev['text_clean'])

    if feat_cols:
        sc = StandardScaler()
        num_tr = csr_matrix(sc.fit_transform(X_tr[feat_cols].values))
        num_te = csr_matrix(sc.transform(X_te[feat_cols].values))
        num_ev = csr_matrix(sc.transform(X_ev[feat_cols].values))
        return (hstack([tfidf_tr, num_tr]),
                hstack([tfidf_te, num_te]),
                hstack([tfidf_ev, num_ev]))
    return tfidf_tr, tfidf_te, tfidf_ev

def fit_eval(name, mat_tr, mat_te, C=0.5, cw=None):
    clf = LinearSVC(C=C, max_iter=1000, random_state=SEED,
                    dual=False, class_weight=cw)
    clf.fit(mat_tr, y_train)
    f1 = f1_score(y_test, clf.predict(mat_te), average='macro')
    delta = f1 - BASELINE_LOCAL
    mark = '✅' if delta > 0 else '⚠️'
    print(f'{mark} {name:<50} F1={f1:.4f}  Δ={delta:+.4f}')
    return f1, clf

print(f'Baseline confirmado v4: F1={BASELINE_LOCAL:.4f}')
print(f'Config base: analyzer=char, (1,4), 400k, min_df=1, all{len(ALL_FEATS)} feats, C=0.5')
print('-' * 70)

Baseline confirmado v4: F1=0.2737
Config base: analyzer=char, (1,4), 400k, min_df=1, all41 feats, C=0.5
----------------------------------------------------------------------


## 7. Exploración de Hiperparámetros

### 7.1 EJE 3 — Filtrado por Frecuencia Mínima (`min_df`)

El parámetro `min_df` controla cuántos documentos debe aparecer un n-grama de caracteres
para ser incluido en el vocabulario:

- **`min_df=1`:** incluye todos los n-gramas, incluso los que aparecen en un solo texto.
  Captura señal específica de cada documento pero también ruido OCR único.
- **`min_df=2`:** requiere que el n-grama aparezca en al menos 2 documentos.
  Filtra artefactos OCR únicos sin perder patrones ortográficos reales.

> ⚠️ **Nota de ejecución:** esta celda forma parte de la exploración original y no se
> re-ejecuta en este notebook. El ganador confirmado fue **`min_df=1`**, que preserva
> más señal temporal al costo de incluir algo de ruido OCR.

In [ ]:
# ══════════════════════════════════════════════════════════
# EJE 3: min_df — ¿filtrar n-gramas únicos ayuda?
# ══════════════════════════════════════════════════════════
# min_df=1: incluye n-gramas que aparecen en UN solo texto
#           muchos son ruido OCR específico de ese documento
# min_df=2: requiere aparecer en al menos 2 documentos
#           filtra el ruido sin perder señal real
print('=== EJE 3: min_df ===')

vec_mdf1 = make_vec(min_df=1, max_f=400000)
tr1, te1, ev1 = build_matrices(vec_mdf1, ALL_FEATS)
f1_mdf1, _ = fit_eval('char(1,4) 400k min_df=1 all_feats [baseline]', tr1, te1)

vec_mdf2 = make_vec(min_df=2, max_f=400000)
tr2, te2, ev2 = build_matrices(vec_mdf2, ALL_FEATS)
f1_mdf2, _ = fit_eval('char(1,4) 400k min_df=2 all_feats',             tr2, te2)

# Ganador de EJE 3
if f1_mdf2 > f1_mdf1:
    BEST_MINDF = 2
    mat_tr_e3, mat_te_e3, mat_ev_e3 = tr2, te2, ev2
    print(f'→ min_df=2 gana')
else:
    BEST_MINDF = 1
    mat_tr_e3, mat_te_e3, mat_ev_e3 = tr1, te1, ev1
    print(f'→ min_df=1 gana')

=== EJE 3: min_df ===


### 7.2 EJE 4 — Subconjunto de Features Numéricas

Se comparan tres configuraciones sobre la mejor config del EJE 3:

- **Sin features numéricas:** solo char TF-IDF — línea base para medir el aporte de las features
- **Top-10 features:** solo las 10 más discriminativas según F-score
- **Todas las features (41):** conjunto completo

> ⚠️ **Nota de ejecución:** esta celda forma parte de la exploración original y no se
> re-ejecuta en este notebook. El ganador confirmado fue **todas las features (41)**,
> superando al top-10 — el conjunto completo aporta señal complementaria que las
> features más relevantes individualmente no capturan solas.

In [ ]:
# ══════════════════════════════════════════════════════════
# EJE 4: features numéricas
# ══════════════════════════════════════════════════════════
# Probamos 3 variantes sobre la mejor config de EJE 3
print('=== EJE 4: features numéricas ===')
print(f'(Usando min_df={BEST_MINDF})')

vec_e4 = make_vec(min_df=BEST_MINDF, max_f=400000)

# Sin features numéricas
tr_none, te_none, ev_none = build_matrices(vec_e4, [])
f1_none, _ = fit_eval('char(1,4) 400k — SIN features numéricas', tr_none, te_none)

# Top-10 features
tr_top10, te_top10, ev_top10 = build_matrices(vec_e4, TOP_10, fit_vec=False)
f1_top10, _ = fit_eval('char(1,4) 400k — top-10 features',       tr_top10, te_top10)

# Todas las features
tr_all, te_all, ev_all = build_matrices(vec_e4, ALL_FEATS, fit_vec=False)
f1_all, _ = fit_eval(f'char(1,4) 400k — todas {len(ALL_FEATS)} features', tr_all, te_all)

# Ganador EJE 4
eje4_scores = {
    'none':  (f1_none,  tr_none,  te_none,  ev_none,  []),
    'top10': (f1_top10, tr_top10, te_top10, ev_top10, TOP_10),
    'all':   (f1_all,   tr_all,   te_all,   ev_all,   ALL_FEATS),
}
best_e4_key = max(eje4_scores, key=lambda k: eje4_scores[k][0])
BEST_F1_E4, mat_tr_best, mat_te_best, mat_ev_best, BEST_FEAT_COLS = eje4_scores[best_e4_key]
print(f'\n→ Ganador EJE 4: {best_e4_key} — F1={BEST_F1_E4:.4f}')

### 7.3 Ajuste Fino del Hiperparámetro C

Se exploran 5 valores de C alrededor del 0.5 conocido como óptimo en versiones anteriores:
`[0.2, 0.3, 0.5, 0.7, 1.0]`. El parámetro C controla la regularización del LinearSVC —
valores más bajos aumentan la regularización y reducen el overfitting en espacios de
alta dimensionalidad como el char TF-IDF de 400k features.

> ⚠️ **Nota de ejecución:** esta celda forma parte de la exploración original y no se
> re-ejecuta en este notebook. El ganador confirmado fue **C=0.5**.

In [ ]:
# ══════════════════════════════════════════════════════════
# AJUSTE FINO DE C
# ══════════════════════════════════════════════════════════
# Solo 5 valores alrededor del 0.5 conocido
# dual=False → usa el solver primal, mucho más rápido con muchas features
print('=== Ajuste fino de C ===')
print(f'(Sobre la mejor config: min_df={BEST_MINDF}, feats={best_e4_key})')

best_C, best_f1_C = 0.5, 0.0
for C in [0.2, 0.3, 0.5, 0.7, 1.0]:
    clf = LinearSVC(C=C, max_iter=1000, random_state=SEED, dual=False)
    clf.fit(mat_tr_best, y_train)
    f1 = f1_score(y_test, clf.predict(mat_te_best), average='macro')
    mark = '  ← MEJOR' if f1 > best_f1_C else ''
    print(f'  C={C}  F1={f1:.4f}{mark}')
    if f1 > best_f1_C:
        best_f1_C, best_C = f1, C

print(f'\nMejor C={best_C} → F1={best_f1_C:.4f}')

### 7.4 Verificación de `class_weight='balanced'`

Con 39 clases casi uniformes (~800 textos por década), no se espera que `class_weight='balanced'`
mejore el desempeño — la corrección de pesos es útil solo cuando hay desbalance significativo.
Se verifica empíricamente en un solo entrenamiento.

> ⚠️ **Nota de ejecución:** esta celda forma parte de la exploración original y no se
> re-ejecuta en este notebook. El ganador confirmado fue **`class_weight=None`** —
> las clases balanceadas no requieren corrección.

In [ ]:
# ══════════════════════════════════════════════════════════
# class_weight='balanced'
# ══════════════════════════════════════════════════════════
# Con 39 clases casi uniformes no debería ayudar,
# pero lo verificamos en un fit
clf_bal = LinearSVC(C=best_C, max_iter=1000, random_state=SEED,
                    dual=False, class_weight='balanced')
clf_bal.fit(mat_tr_best, y_train)
f1_bal = f1_score(y_test, clf_bal.predict(mat_te_best), average='macro')

print(f'class_weight=balanced: F1={f1_bal:.4f}')
print(f'class_weight=None:     F1={best_f1_C:.4f}')

FINAL_CW = 'balanced' if f1_bal > best_f1_C else None
FINAL_F1 = max(f1_bal, best_f1_C)
print(f'→ Usar class_weight={FINAL_CW}  (F1 final local: {FINAL_F1:.4f})')

## 8. Resumen de Experimentos

Antes del entrenamiento final, consolidamos los resultados de toda la exploración
de hiperparámetros realizada en versiones anteriores del notebook:

| Eje | Parámetro explorado | Ganador | Justificación |
|---|---|---|---|
| EJE 3 | `min_df` | `min_df=1` | Preserva más señal temporal |
| EJE 4 | Features numéricas | Todas (41) | El conjunto completo supera al top-10 |
| Ajuste C | Regularización | `C=0.5` | Óptimo en espacio de 400k features |
| Class weight | Balanceo de clases | `None` | 39 clases ya balanceadas |

La configuración ganadora completa es:
- **Char TF-IDF:** `analyzer='char'`, `ngram_range=(1,4)`, `max_features=400k`,
  `min_df=1`, `sublinear_tf=True`, `strip_accents=None`
- **Features numéricas:** las 41 features históricas completas
- **Clasificador:** `LinearSVC(C=0.5, dual=False, class_weight=None)`
- **F1-macro local (v4):** 0.2737
- **Kaggle (v3 individual):** 0.290

In [ ]:
# ══════════════════════════════════════════════════════════
# RESUMEN DE EXPERIMENTOS
# ══════════════════════════════════════════════════════════
print('=' * 60)
print('RESUMEN DE EXPERIMENTOS v5')
print('=' * 60)
print(f'Baseline v4 (local):   {BASELINE_LOCAL:.4f}')
print(f'Kaggle actual:         {BASELINE_KAGGLE:.4f}')
print()
print(f'EJE 3 — min_df ganador:  {BEST_MINDF}')
print(f'EJE 4 — feats ganador:   {best_e4_key} ({len(BEST_FEAT_COLS)} features)')
print(f'C óptimo:                {best_C}')
print(f'class_weight:            {FINAL_CW}')
print()
print(f'F1 local final:          {FINAL_F1:.4f}')
print(f'Mejora sobre baseline:   {FINAL_F1 - BASELINE_LOCAL:+.4f}')
print(f'Proyección Kaggle:      ~{BASELINE_KAGGLE + (FINAL_F1 - BASELINE_LOCAL):.4f}')
print('=' * 60)

## 9. Entrenamiento Final

### 9.1 Modelo Base (configuración individual)

Este bloque corresponde al entrenamiento final original con la configuración ganadora
de la exploración: **char TF-IDF (400k) + 41 features numéricas + LinearSVC**.
Se entrena con el 100% de los datos para maximizar la señal disponible.

> ⚠️ Esta celda está presente como documentación del proceso. El modelo combinado
> de la Sección 9.2 supera este resultado y es el que se usa para el envío final.

In [ ]:
# ══════════════════════════════════════════════════════════
# ENTRENAMIENTO FINAL — 100% DE LOS DATOS
# ══════════════════════════════════════════════════════════
print('Entrenando modelo final con 100% de los datos...')
print(f'  analyzer:    {BEST_ANALYZER}')
print(f'  ngram:       {BEST_NGRAM}')
print(f'  max_features:{BEST_MAXF}')
print(f'  min_df:      {BEST_MINDF}')
print(f'  feat_cols:   {len(BEST_FEAT_COLS)} features ({best_e4_key})')
print(f'  C:           {best_C}')
print(f'  dual:        False')
print(f'  class_weight:{FINAL_CW}')

X_full = pd.concat([
    data['text_clean'].reset_index(drop=True),
    feats_train.reset_index(drop=True)
], axis=1)
y_full = data['decade'].reset_index(drop=True)

# Vectorizador final
vec_final = make_vec(min_df=BEST_MINDF, max_f=BEST_MAXF)
tfidf_full = vec_final.fit_transform(X_full['text_clean'])
tfidf_eval = vec_final.transform(X_eval_full['text_clean'])

# Features numéricas
if BEST_FEAT_COLS:
    sc_final   = StandardScaler()
    num_full   = csr_matrix(sc_final.fit_transform(X_full[BEST_FEAT_COLS].values))
    num_eval   = csr_matrix(sc_final.transform(X_eval_full[BEST_FEAT_COLS].values))
    X_full_mat = hstack([tfidf_full, num_full])
    X_eval_mat = hstack([tfidf_eval, num_eval])
else:
    X_full_mat = tfidf_full
    X_eval_mat = tfidf_eval

# Modelo final
clf_final = LinearSVC(C=best_C, max_iter=1000, random_state=SEED,
                      dual=False, class_weight=FINAL_CW)
clf_final.fit(X_full_mat, y_full)
print('✅ Modelo final entrenado')

In [ ]:
# ── Submission ──
preds = clf_final.predict(X_eval_mat)

sub = pd.DataFrame({'id': df_eval['id'], 'answer': preds})
sub.to_csv('./submissions/SUBMISSION_v5_FINAL.csv', index=False)

print('✅ SUBMISSION_v5_FINAL.csv generado')
print(f'Predicciones: {len(preds)} | Clases únicas: {len(set(preds))}')
print()
print('═' * 60)
print('RESUMEN FINAL')
print('═' * 60)
print(f'Score Kaggle anterior:   {BASELINE_KAGGLE}')
print(f'F1 local v5:             {FINAL_F1:.4f}')
print(f'Mejora local:            {FINAL_F1 - BASELINE_LOCAL:+.4f}')
print(f'Proyección Kaggle:      ~{BASELINE_KAGGLE + (FINAL_F1 - BASELINE_LOCAL):.4f}')
print('═' * 60)
print()
print('→ SUBIR: SUBMISSION_v5_FINAL.csv')

In [10]:
# ── Variables ganadoras de la exploración (v4/v5) ──
BEST_MINDF     = 1
BEST_FEAT_COLS = ALL_FEATS
best_C         = 0.5
FINAL_CW       = None
best_e4_key    = 'all'

print('✅ Variables ganadoras definidas')
print(f'  min_df:      {BEST_MINDF}')
print(f'  feat_cols:   {len(BEST_FEAT_COLS)} features')
print(f'  C:           {best_C}')
print(f'  class_weight:{FINAL_CW}')

✅ Variables ganadoras definidas
  min_df:      1
  feat_cols:   41 features
  C:           0.5
  class_weight:None


### 9.2 Modelo Combinado — Mejor Resultado del Equipo

El modelo combinado agrega **Word TF-IDF (1,2) con 100k features** a la configuración
ganadora del compañero, produciendo una representación de **500.041 features** que
captura señal complementaria a nivel de palabra y a nivel de carácter simultáneamente:

| Representación | Configuración | Features | Señal capturada |
|---|---|---|---|
| Char TF-IDF | `(1,4)`, 400k, `min_df=1` | 400.000 | Ortografía sub-léxica, dígrafos históricos, ruido OCR robusto |
| Word TF-IDF | `(1,2)`, 100k, `sublinear_tf` | 100.000 | Vocabulario histórico, bigramas temporales |
| Features numéricas | 41 features históricas | 41 | Morfología, puntuación, patrones regex arcaicos |

**Evolución del score en Kaggle:**

| Versión | Estrategia | Kaggle Accuracy |
|---|---|---|
| Baseline competencia | — | 0.196 |
| Envío 3 (individual) | LR + features numéricas | 0.225 |
| Envío 6 (individual) | LR + word 100k | 0.230 |
| Envío 10 (individual) | LR + word 100k + 100% datos | 0.241 |
| v3 compañero | char 400k + 41 features + LinearSVC | 0.290 |
| Combinado (este notebook) | char 400k + word 100k + 41 features | **0.306 ⭐** |

La ganancia de **+0.016** sobre el mejor resultado individual viene de combinar dos
representaciones ortogonales: el char TF-IDF captura patrones sub-léxicos robustos
al ruido OCR, mientras que el word TF-IDF captura el vocabulario histórico completo
que el análisis de caracteres no puede representar directamente.

In [11]:
# ══════════════════════════════════════════════════════════
# ENTRENAMIENTO FINAL — 100% DE LOS DATOS
# char TF-IDF (suyo) + word TF-IDF (nuestro) + features numéricas
# ══════════════════════════════════════════════════════════
print('Entrenando modelo final con 100% de los datos...')
print(f'  analyzer:    {BEST_ANALYZER}')
print(f'  ngram:       {BEST_NGRAM}')
print(f'  max_features:{BEST_MAXF}')
print(f'  min_df:      {BEST_MINDF}')
print(f'  feat_cols:   {len(BEST_FEAT_COLS)} features ({best_e4_key})')
print(f'  C:           {best_C}')
print(f'  dual:        False')
print(f'  class_weight:{FINAL_CW}')
print(f'  word TF-IDF: ngram=(1,2), max_features=100k  ← AGREGADO')

X_full = pd.concat([
    data['text_clean'].reset_index(drop=True),
    feats_train.reset_index(drop=True)
], axis=1)
y_full = data['decade'].reset_index(drop=True)

# ── Char TF-IDF (suyo, intacto) ──
vec_final  = make_vec(min_df=BEST_MINDF, max_f=BEST_MAXF)
tfidf_full = vec_final.fit_transform(X_full['text_clean'])
tfidf_eval = vec_final.transform(X_eval_full['text_clean'])

# ── Word TF-IDF (nuestro, agregado) ──
word_vec   = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=100_000,
    sublinear_tf=True,
)
word_full  = word_vec.fit_transform(X_full['text_clean'])
word_eval  = word_vec.transform(X_eval_full['text_clean'])

print(f'\nShapes:')
print(f'  char TF-IDF:  {tfidf_full.shape}')
print(f'  word TF-IDF:  {word_full.shape}')

# ── Features numéricas (suyas, intactas) ──
if BEST_FEAT_COLS:
    sc_final   = StandardScaler()
    num_full   = csr_matrix(sc_final.fit_transform(X_full[BEST_FEAT_COLS].values))
    num_eval   = csr_matrix(sc_final.transform(X_eval_full[BEST_FEAT_COLS].values))
    X_full_mat = hstack([tfidf_full, word_full, num_full])
    X_eval_mat = hstack([tfidf_eval, word_eval, num_eval])
else:
    X_full_mat = hstack([tfidf_full, word_full])
    X_eval_mat = hstack([tfidf_eval, word_eval])

print(f'  combinado:    {X_full_mat.shape}')

# ── Modelo final (suyo, intacto) ──
clf_final = LinearSVC(C=best_C, max_iter=1000, random_state=SEED,
                      dual=False, class_weight=FINAL_CW)
clf_final.fit(X_full_mat, y_full)
print('✅ Modelo final entrenado')

Entrenando modelo final con 100% de los datos...
  analyzer:    char
  ngram:       (1, 4)
  max_features:400000
  min_df:      1
  feat_cols:   41 features (all)
  C:           0.5
  dual:        False
  class_weight:None
  word TF-IDF: ngram=(1,2), max_features=100k  ← AGREGADO

Shapes:
  char TF-IDF:  (31352, 400000)
  word TF-IDF:  (31352, 100000)
  combinado:    (31352, 500041)
✅ Modelo final entrenado


## 10. Generación de Predicciones Finales

Se generan las predicciones sobre los **3.490 textos** de `eval.csv` con el modelo
combinado. Los transformadores se aplican usando únicamente `transform` — nunca
`fit_transform` — para garantizar consistencia con la representación de entrenamiento.

El archivo `SUBMISSION_v5_word_char.csv` obtuvo un **Kaggle Accuracy de 0.30563**,
el mejor resultado del equipo en la competencia.

In [12]:
# ── Submission ──
preds = clf_final.predict(X_eval_mat)

sub = pd.DataFrame({'id': df_eval['id'], 'answer': preds})
sub.to_csv('./submissions/SUBMISSION_v5_word_char.csv', index=False)

print('✅ SUBMISSION_v5_word_char.csv generado')
print(f'Predicciones: {len(preds)} | Clases únicas: {len(set(preds))}')

✅ SUBMISSION_v5_word_char.csv generado
Predicciones: 3490 | Clases únicas: 39


## 11. Guardado del Modelo

Se guardan con `joblib` todos los componentes necesarios para reproducir predicciones
en producción sin necesidad de reentrenar: el vectorizador char TF-IDF, el vectorizador
word TF-IDF, el scaler de features numéricas y el clasificador final.

In [12]:
# ─────────────────────────────────────────────
# 11. GUARDADO DEL MODELO
# ─────────────────────────────────────────────
import joblib

os.makedirs('./model', exist_ok=True)

joblib.dump(vec_final,  './model/vec_char.pkl')
joblib.dump(word_vec,   './model/vec_word.pkl')
joblib.dump(sc_final,   './model/scaler.pkl')
joblib.dump(clf_final,  './model/clf_final.pkl')

print('✅ Modelo guardado en ./model/')
print('   vec_char.pkl  — char TF-IDF (400k features)')
print('   vec_word.pkl  — word TF-IDF (100k features)')
print('   scaler.pkl    — StandardScaler (41 features)')
print('   clf_final.pkl — LinearSVC (C=0.5, dual=False)')
print()
print('Para cargar y predecir:')
print('   vec_char = joblib.load("./model/vec_char.pkl")')
print('   vec_word = joblib.load("./model/vec_word.pkl")')
print('   scaler   = joblib.load("./model/scaler.pkl")')
print('   clf      = joblib.load("./model/clf_final.pkl")')

✅ Modelo guardado en ./model/
   vec_char.pkl  — char TF-IDF (400k features)
   vec_word.pkl  — word TF-IDF (100k features)
   scaler.pkl    — StandardScaler (41 features)
   clf_final.pkl — LinearSVC (C=0.5, dual=False)

Para cargar y predecir:
   vec_char = joblib.load("./model/vec_char.pkl")
   vec_word = joblib.load("./model/vec_word.pkl")
   scaler   = joblib.load("./model/scaler.pkl")
   clf      = joblib.load("./model/clf_final.pkl")
